# Train LightGBM Model

In [5]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr
from lightgbm import LGBMRegressor      # pip install lightgbm

data = pd.read_parquet('../data/processed/data_with_listeners.parquet')
orig_data= pd.read_parquet('../data/processed/orig_data.parquet')
target = 'popularity'

# ---- split ONCE; both stages use the same test set so it's never touched in training
train, test = train_test_split(data, test_size=0.2, random_state=42)

In [8]:
# ===== RAW popularity model =====
from sklearn.model_selection import GridSearchCV

target = 'popularity'
audio = ['energy','danceability','loudness','speechiness','instrumentalness',
         'acousticness','valence','tempo','liveness','duration_ms']   # <-- your actual audio cols
feats = audio + ['artists_listeners', 'track_genre']   # fame INCLUDED — see note below

X_tr, X_te = train[feats].copy(), test[feats].copy()
X_tr['track_genre'] = X_tr['track_genre'].astype('category')
X_te['track_genre'] = X_te['track_genre'].astype('category')
y_tr, y_te = train[target], test[target]

param_grid = {'n_estimators':[100,200,300], 'learning_rate': list(np.arange(0.02,0.13,0.03))}
gs = GridSearchCV(LGBMRegressor(verbose=-1, objective='tweedie'), param_grid, scoring='r2', cv=10, n_jobs=-1)
gs.fit(X_tr, y_tr)
print("best params:", gs.best_params_)

model = LGBMRegressor(**gs.best_params_, random_state=667, verbose=-1)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)

print("R²      :", r2_score(y_te, pred))
print("MAE     :", mean_absolute_error(y_te, pred))
print("Spearman:", spearmanr(y_te, pred).correlation)

print(pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False))  # the payoff

best params: {'learning_rate': np.float64(0.11000000000000001), 'n_estimators': 300}
R²      : 0.706736849130829
MAE     : 6.536875368505609
Spearman: 0.8411236551363582
track_genre          1863
artists_listeners    1614
duration_ms           618
danceability          603
tempo                 579
loudness              569
valence               560
acousticness          555
speechiness           534
instrumentalness      522
energy                512
liveness              471
dtype: int32


In [9]:
from sklearn.metrics import root_mean_squared_error
print("ctx RMSE :", root_mean_squared_error(y_te, pred))

ctx RMSE : 9.571463449136187


# Train HistGradientBoostingRegressor

In [10]:
# Clean grouped evaluation setup used by the model cells below.
import pandas as pd
import numpy as np

from scipy.stats import loguniform, randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 667
TARGET = "popularity"
GROUP_COLUMN = "primary_artist"
CV_SPLITS = 3
N_ITER_HGB = 15
N_ITER_FOREST = 6
N_ITER_XGB = 12
SEARCH_N_JOBS = -1

audio_features = [
    "energy",
    "danceability",
    "loudness",
    "speechiness",
    "instrumentalness",
    "acousticness",
    "valence",
    "tempo",
    "liveness",
    "duration_ms",
    "key",
    "mode",
    "time_signature",
]
numeric_features = audio_features + ["log_artists_listeners"]
categorical_features = ["explicit", "track_genre"]

df = pd.read_parquet("../data/processed/orig_data_with_listeners.parquet").copy()
df["log_artists_listeners"] = np.log1p(df["artists_listeners"])

needed_columns = [TARGET, GROUP_COLUMN] + numeric_features + categorical_features
model_df = df[needed_columns].dropna().reset_index(drop=True)

X = model_df[numeric_features + categorical_features]
y = model_df[TARGET]
groups = model_df[GROUP_COLUMN]

holdout_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(holdout_split.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_train = groups.iloc[train_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

group_cv = GroupKFold(n_splits=CV_SPLITS)
model_results = []

dense_preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

def fit_search(name, estimator, param_distributions, n_iter):
    pipe = Pipeline([
        ("preprocess", dense_preprocess),
        ("model", estimator),
    ])

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring=scoring,
        refit="rmse",
        cv=group_cv,
        n_jobs=SEARCH_N_JOBS,
        random_state=RANDOM_STATE,
        verbose=1,
        return_train_score=False,
        error_score="raise",
    )
    search.fit(X_train, y_train, groups=groups_train)

    predictions = search.best_estimator_.predict(X_test)
    result = {
        "model": name,
        "cv_rmse": -search.best_score_,
        "cv_r2": search.cv_results_["mean_test_r2"][search.best_index_],
        "holdout_rmse": root_mean_squared_error(y_test, predictions),
        "holdout_mae": mean_absolute_error(y_test, predictions),
        "holdout_r2": r2_score(y_test, predictions),
    }
    model_results.append(result)

    print(f"{name} best params:")
    print(search.best_params_)
    display(pd.DataFrame([result]).round(4))
    return search, result

hist_estimator = HistGradientBoostingRegressor(
    loss="squared_error",
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=RANDOM_STATE,
)

hist_params = {
    "model__max_iter": randint(150, 700),
    "model__learning_rate": loguniform(0.02, 0.2),
    "model__max_leaf_nodes": randint(15, 64),
    "model__min_samples_leaf": randint(20, 150),
    "model__l2_regularization": loguniform(1e-4, 10.0),
}

print(f"Rows: {len(model_df):,} | train: {len(X_train):,} | holdout: {len(X_test):,}")
print(f"Train artists: {groups_train.nunique():,} | holdout artists: {groups_test.nunique():,}")
hist_search, hist_result = fit_search(
    "HistGradientBoostingRegressor",
    hist_estimator,
    hist_params,
    n_iter=N_ITER_HGB,
)


Rows: 66,804 | train: 53,982 | holdout: 12,822
Train artists: 13,391 | holdout artists: 3,348
Fitting 3 folds for each of 15 candidates, totalling 45 fits
HistGradientBoostingRegressor best params:
{'model__l2_regularization': np.float64(0.10367504953145981), 'model__learning_rate': np.float64(0.07132004887554133), 'model__max_iter': 345, 'model__max_leaf_nodes': 16, 'model__min_samples_leaf': 40}


,model,cv_rmse,cv_r2,holdout_rmse,holdout_mae,holdout_r2
0,HistGradientBoostingRegressor,10.704,0.6321,10.8151,7.7801,0.6401


# Train RandomForestRegressor

In [11]:
from sklearn.ensemble import RandomForestRegressor

rf_estimator = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1,
    bootstrap=True,
)

rf_params = {
    "model__n_estimators": randint(150, 500),
    "model__max_depth": [None, 8, 12, 16, 24, 32],
    "model__min_samples_leaf": randint(1, 20),
    "model__min_samples_split": randint(2, 30),
    "model__max_features": ["sqrt", "log2", 0.5, 0.75, 1.0],
    "model__max_samples": uniform(0.5, 0.5),
}

rf_search, rf_result = fit_search(
    "RandomForestRegressor",
    rf_estimator,
    rf_params,
    n_iter=N_ITER_FOREST,
)

display(pd.DataFrame(model_results).round(4).sort_values("holdout_rmse"))


Fitting 3 folds for each of 6 candidates, totalling 18 fits
RandomForestRegressor best params:
{'model__max_depth': None, 'model__max_features': 0.5, 'model__max_samples': np.float64(0.9854257356857004), 'model__min_samples_leaf': 16, 'model__min_samples_split': 24, 'model__n_estimators': 468}


,model,cv_rmse,cv_r2,holdout_rmse,holdout_mae,holdout_r2
0,RandomForestRegressor,11.6243,0.5662,11.5921,8.3391,0.5865


,model,cv_rmse,cv_r2,holdout_rmse,holdout_mae,holdout_r2
0,HistGradientBoostingRegressor,10.7040,0.6321,10.8151,7.7801,0.6401
1,RandomForestRegressor,11.6243,0.5662,11.5921,8.3391,0.5865


# Train ExtraTreesRegressor

In [12]:
from sklearn.ensemble import ExtraTreesRegressor

extra_estimator = ExtraTreesRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1,
    bootstrap=False,
)

extra_params = {
    "model__n_estimators": randint(200, 700),
    "model__max_depth": [None, 8, 12, 16, 24, 32],
    "model__min_samples_leaf": randint(1, 20),
    "model__min_samples_split": randint(2, 30),
    "model__max_features": ["sqrt", "log2", 0.5, 0.75, 1.0],
}

extra_search, extra_result = fit_search(
    "ExtraTreesRegressor",
    extra_estimator,
    extra_params,
    n_iter=N_ITER_FOREST,
)

display(pd.DataFrame(model_results).round(4).sort_values("holdout_rmse"))


Fitting 3 folds for each of 6 candidates, totalling 18 fits
ExtraTreesRegressor best params:
{'model__max_depth': None, 'model__max_features': 0.5, 'model__min_samples_leaf': 3, 'model__min_samples_split': 20, 'model__n_estimators': 567}


,model,cv_rmse,cv_r2,holdout_rmse,holdout_mae,holdout_r2
0,ExtraTreesRegressor,10.837,0.623,10.634,7.3775,0.6521


,model,cv_rmse,cv_r2,holdout_rmse,holdout_mae,holdout_r2
2,ExtraTreesRegressor,10.8370,0.6230,10.6340,7.3775,0.6521
0,HistGradientBoostingRegressor,10.7040,0.6321,10.8151,7.7801,0.6401
1,RandomForestRegressor,11.6243,0.5662,11.5921,8.3391,0.5865


# Implement XGBoost

In [13]:
try:
    from xgboost import XGBRegressor
except ImportError:
    print("xgboost is not installed. Install it in this environment, then rerun this cell.")
    xgb_search = None
else:
    xgb_estimator = XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        eval_metric="rmse",
        random_state=RANDOM_STATE,
        n_jobs=1,
    )

    xgb_params = {
        "model__n_estimators": randint(200, 900),
        "model__learning_rate": loguniform(0.01, 0.2),
        "model__max_depth": randint(3, 10),
        "model__min_child_weight": loguniform(0.5, 20.0),
        "model__subsample": uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.6, 0.4),
        "model__reg_alpha": loguniform(1e-4, 10.0),
        "model__reg_lambda": loguniform(0.1, 20.0),
    }

    xgb_search, xgb_result = fit_search(
        "XGBRegressor",
        xgb_estimator,
        xgb_params,
        n_iter=N_ITER_XGB,
    )

    display(pd.DataFrame(model_results).round(4).sort_values("holdout_rmse"))


Fitting 3 folds for each of 12 candidates, totalling 36 fits
XGBRegressor best params:
{'model__colsample_bytree': np.float64(0.6924348339178141), 'model__learning_rate': np.float64(0.11082016266500137), 'model__max_depth': 8, 'model__min_child_weight': np.float64(4.54148538095132), 'model__n_estimators': 325, 'model__reg_alpha': np.float64(0.510954543926005), 'model__reg_lambda': np.float64(10.820603174905312), 'model__subsample': np.float64(0.8961682344337496)}


,model,cv_rmse,cv_r2,holdout_rmse,holdout_mae,holdout_r2
0,XGBRegressor,10.6031,0.639,10.6056,7.4617,0.6539


,model,cv_rmse,cv_r2,holdout_rmse,holdout_mae,holdout_r2
3,XGBRegressor,10.6031,0.6390,10.6056,7.4617,0.6539
2,ExtraTreesRegressor,10.8370,0.6230,10.6340,7.3775,0.6521
0,HistGradientBoostingRegressor,10.7040,0.6321,10.8151,7.7801,0.6401
1,RandomForestRegressor,11.6243,0.5662,11.5921,8.3391,0.5865
